# 01 — Data Profiling

Notebook ini melakukan eksplorasi awal dataset **`idn-news-az`** menggunakan
**Polars Lazy API**. Tujuannya adalah membuktikan skala dataset, memahami
struktur kolom, memeriksa kualitas awal, serta menyiapkan dasar untuk tahap
cleaning.

Profiling mencakup:
- inventaris file dan ukuran dataset,
- jumlah baris dan kolom,
- schema dan contoh data,
- missing values,
- nilai unik dan potensi duplikasi,
- validasi tanggal,
- panjang judul dan isi artikel,
- distribusi domain sumber berita.

In [32]:
import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

print("Polars version:", pl.__version__)

Polars version: 1.44.2


## 1. Lokasi dataset

In [33]:
from pathlib import Path

# Notebook dapat dijalankan baik dari root project maupun dari folder notebooks/
CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Root project tidak ditemukan. Jalankan notebook dari repository tugas."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "idn-news-az" / "data_files"
RAW_GLOB = str(RAW_DIR / "*.parquet")

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Folder dataset tidak ditemukan: {RAW_DIR}")

PARQUET_FILES = sorted(RAW_DIR.glob("*.parquet"))
if not PARQUET_FILES:
    raise FileNotFoundError(f"Tidak ada file .parquet di {RAW_DIR}")

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("Jumlah file  :", len(PARQUET_FILES))

PROJECT_ROOT : C:\Users\endo1\Documents\SEMESTER  7\big data\tugas1-MohammadRavlindoS
RAW_DIR      : C:\Users\endo1\Documents\SEMESTER  7\big data\tugas1-MohammadRavlindoS\data\raw\idn-news-az\data_files
Jumlah file  : 441


## 2. Ukuran dataset

In [34]:
total_size_bytes = sum(p.stat().st_size for p in PARQUET_FILES)
total_size_mb = total_size_bytes / (1024 ** 2)
total_size_gb = total_size_bytes / (1024 ** 3)

print(f"Jumlah file Parquet : {len(PARQUET_FILES):,}")
print(f"Ukuran dataset      : {total_size_mb:,.2f} MB")
print(f"Ukuran dataset      : {total_size_gb:,.2f} GB")

Jumlah file Parquet : 441
Ukuran dataset      : 1,486.16 MB
Ukuran dataset      : 1.45 GB


## 3. Membaca dataset dengan Lazy Evaluation

In [35]:
lf = pl.scan_parquet(RAW_GLOB)

# LazyFrame: data belum dimuat penuh ke memori.
lf

## 4. Schema dan contoh data

In [36]:
schema = lf.collect_schema()
columns = schema.names()

print(f"Jumlah kolom: {len(columns)}")
for name, dtype in schema.items():
    print(f"- {name}: {dtype}")

Jumlah kolom: 4
- date: String
- link: String
- title: String
- text: String


In [37]:
lf.head(10).collect()

date,link,title,text
str,str,str,str
"""2019-05-10""","""https://www.cnbcindonesia.com/…","""Potret Jemaah Haji Mulai Melak…","""Haji tahun ini merupakan tanta…"
"""2019-08-19""","""https://www.cnbcindonesia.com/…","""Cek Warga China Mencoba Robot …","""Seorang anggota staf mengambil…"
"""2019-08-26""","""https://www.cnbcindonesia.com/…","""Ajaib! Tato Elektronik Buatan …","""""Di masa depan, apa yang kami …"
"""2019-10-30""","""https://www.cnbcindonesia.com/…","""Chairul Tanjung Hadiri BSI GIF…","""Menurutnya, karakter anak-anak…"
"""2022-03-28""","""https://www.cnbcindonesia.com/…","""11 Tanda Rumah Anda Sudah Terc…","""Jakarta, CNBC Indonesia - Polu…"
"""2022-04-05""","""https://www.cnbcindonesia.com/…","""Warga RI Pakai Baju Koko Buat …","""Jakarta, CNBC Indonesia - Mesk…"
"""2022-04-05""","""https://www.cnbcindonesia.com/…","""Asal-Usul Peci: Dipakai Shalat…","""Jakarta, CNBC Indonesia - Di I…"
"""2022-04-20""","""https://www.cnbcindonesia.com/…","""Jilbab di RI: Sempat Dilarang …","""Jakarta, CNBC Indonesia - Saat…"
"""2022-06-08""","""https://www.cnbcindonesia.com/…","""Diserbu Produk Impor, Pengusah…","""Jakarta, CNBC Indonesia- Kerag…"


## 5. Jumlah baris dan kolom

In [38]:
total_rows = lf.select(pl.len().alias("jumlah_baris")).collect().item()
total_columns = len(columns)

summary = pl.DataFrame({
    "metrik": [
        "jumlah_file",
        "ukuran_mb",
        "ukuran_gb",
        "jumlah_baris",
        "jumlah_kolom",
    ],
    "nilai": [
        float(len(PARQUET_FILES)),
        round(total_size_mb, 2),
        round(total_size_gb, 2),
        float(total_rows),
        float(total_columns),
    ],
})

print(f"Jumlah baris : {total_rows:,}")
print(f"Jumlah kolom : {total_columns}")
summary

Jumlah baris : 1,149,789
Jumlah kolom : 4


metrik,nilai
str,f64
"""jumlah_file""",441.0
"""ukuran_mb""",1486.16
"""ukuran_gb""",1.45
"""jumlah_baris""",1.149789e6
"""jumlah_kolom""",4.0


## 6. Missing values

In [39]:
null_counts = (
    lf.select([
        pl.col(c).null_count().alias(c)
        for c in columns
    ])
    .collect()
)

null_profile = (
    null_counts
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"],
    )
    .with_columns(
        (pl.col("jumlah_null") / total_rows * 100)
        .round(4)
        .alias("persentase_null")
    )
    .sort("jumlah_null", descending=True)
)

null_profile

kolom,jumlah_null,persentase_null
str,u32,f64
"""date""",0,0.0
"""link""",0,0.0
"""title""",0,0.0
"""text""",0,0.0


## 7. Nilai unik pada kolom penting

In [40]:
important_columns = [c for c in ["date", "link", "title"] if c in columns]

unique_profile = lf.select([
    pl.col(c).n_unique().alias(c)
    for c in important_columns
]).collect()

unique_profile

date,link,title
u32,u32,u32
441,1029485,1024495


## 8. Potensi duplikasi

In [41]:
duplicate_summary = {}

if "link" in columns:
    duplicate_summary["duplicate_link"] = (
        lf.filter(pl.col("link").is_not_null())
        .select((pl.len() - pl.col("link").n_unique()).alias("duplicates"))
        .collect()
        .item()
    )

if "title" in columns:
    duplicate_summary["duplicate_title"] = (
        lf.filter(pl.col("title").is_not_null())
        .select((pl.len() - pl.col("title").n_unique()).alias("duplicates"))
        .collect()
        .item()
    )

duplicate_summary

{'duplicate_link': 120304, 'duplicate_title': 125294}

> Catatan: pada tahap profiling, baris belum dihapus. Duplikasi baru ditangani
> pada notebook `02_data_cleaning.ipynb`.

## 9. Validasi dan rentang tanggal

In [42]:
if "date" not in columns:
    raise KeyError("Kolom 'date' tidak ditemukan pada dataset.")

lf_profile = lf.with_columns(
    pl.col("date")
    .cast(pl.String)
    .str.to_datetime(strict=False, exact=False)
    .alias("date_parsed")
)

date_profile = lf_profile.select([
    pl.col("date_parsed").min().alias("tanggal_awal"),
    pl.col("date_parsed").max().alias("tanggal_akhir"),
    pl.col("date_parsed").null_count().alias("tanggal_tidak_valid"),
]).collect()

date_profile

tanggal_awal,tanggal_akhir,tanggal_tidak_valid
datetime[μs],datetime[μs],u32
2019-05-10 00:00:00,2023-10-01 00:00:00,101421


## 10. Statistik panjang judul dan artikel

In [43]:
length_exprs = []
if "title" in columns:
    length_exprs += [
        pl.col("title").cast(pl.String).str.len_chars().min().alias("title_min"),
        pl.col("title").cast(pl.String).str.len_chars().mean().round(2).alias("title_mean"),
        pl.col("title").cast(pl.String).str.len_chars().median().alias("title_median"),
        pl.col("title").cast(pl.String).str.len_chars().max().alias("title_max"),
    ]

if "text" in columns:
    length_exprs += [
        pl.col("text").cast(pl.String).str.len_chars().min().alias("text_min"),
        pl.col("text").cast(pl.String).str.len_chars().mean().round(2).alias("text_mean"),
        pl.col("text").cast(pl.String).str.len_chars().median().alias("text_median"),
        pl.col("text").cast(pl.String).str.len_chars().max().alias("text_max"),
    ]

length_stats = lf.select(length_exprs).collect()
length_stats

title_min,title_mean,title_median,title_max,text_min,text_mean,text_median,text_max
u32,f64,f64,u32,u32,f64,f64,u32
8,70.5,69.0,199,14,2370.58,2148.0,71048


## 11. Distribusi domain sumber berita

In [44]:
if "link" not in columns:
    raise KeyError("Kolom 'link' tidak ditemukan pada dataset.")

source_profile = (
    lf.with_columns(
        pl.col("link")
        .cast(pl.String)
        .str.extract(r"https?://(?:www\.)?([^/]+)", 1)
        .str.to_lowercase()
        .alias("source_domain")
    )
    .group_by("source_domain")
    .agg(pl.len().alias("jumlah_artikel"))
    .sort("jumlah_artikel", descending=True)
    .head(20)
    .collect()
)

source_profile

source_domain,jumlah_artikel
str,u32
"""tribunnews.com""",176899
"""cnnindonesia.com""",101421
"""cnbcindonesia.com""",89008
"""jpnn.com""",88117
"""news.detik.com""",79662
"""investor.id""",38627
"""kompas.com""",30560
"""antaranews.com""",20209
"""kabar24.bisnis.com""",20087


## 12. Jumlah artikel per tahun

In [45]:
articles_per_year = (
    lf_profile
    .filter(pl.col("date_parsed").is_not_null())
    .with_columns(pl.col("date_parsed").dt.year().alias("tahun"))
    .group_by("tahun")
    .agg(pl.len().alias("jumlah_artikel"))
    .sort("tahun")
    .collect()
)

articles_per_year

tahun,jumlah_artikel
i32,u32
2019,4
2022,148090
2023,900274


## 13. Ringkasan kelayakan dataset

In [46]:
meets_rows = total_rows > 1_000_000
meets_size = total_size_mb >= 500

print("=== RINGKASAN PROFILING ===")
print(f"Jumlah file        : {len(PARQUET_FILES):,}")
print(f"Ukuran dataset     : {total_size_gb:.2f} GB")
print(f"Jumlah baris       : {total_rows:,}")
print(f"Jumlah kolom       : {total_columns}")
print(f"> 1 juta baris     : {'YA' if meets_rows else 'TIDAK'}")
print(f">= 500 MB          : {'YA' if meets_size else 'TIDAK'}")
print(f"Memenuhi syarat    : {'YA' if (meets_rows or meets_size) else 'TIDAK'}")

=== RINGKASAN PROFILING ===
Jumlah file        : 441
Ukuran dataset     : 1.45 GB
Jumlah baris       : 1,149,789
Jumlah kolom       : 4
> 1 juta baris     : YA
>= 500 MB          : YA
Memenuhi syarat    : YA


## Kesimpulan Profiling

Dataset dibaca menggunakan `scan_parquet()` sehingga proses awal memanfaatkan
lazy evaluation. Hasil profiling di atas digunakan sebagai dasar untuk
menentukan aturan cleaning pada notebook berikutnya.

Hal yang perlu diperhatikan pada tahap cleaning:
1. nilai null pada kolom penting,
2. tanggal yang gagal diparsing,
3. artikel dengan URL yang sama,
4. panjang judul/isi yang ekstrem,
5. konsistensi domain sumber berita.